## What to Vary

In [ ]:
# language="english"
# language="multilingual"
# DeepPavlov/rubert-base-cased-sentence


# raw text or vw text


# default topics (whatever)
# specific number of topics


# KeyBERTInspired
# openchat

In [19]:
from topicnet.cooking_machine import Dataset

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, TextGeneration

from umap import UMAP
from hdbscan import HDBSCAN

from hdbscan.flat import HDBSCAN_flat

from sklearn.feature_extraction.text import CountVectorizer

import gensim.corpora as corpora

from gensim.models.coherencemodel import CoherenceModel

import pandas as pd

In [20]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [21]:
! ls $DATA_FOLDER_PATH

20NG.csv	 MKB10__internals      RTL_Wiki_person.csv
20NG__internals  postnauka.csv	       RTL_Wiki_person__internals
api.py		 postnauka__internals  ruwiki_good__internals
Brown		 __pycache__	       ruwiki_good.txt
Brown_BOW.csv	 Reuters	       WikiRef-220
Brown_NOOW.csv	 Reuters_BOW.csv       wiki_ref220_bow.csv
__init__.py	 Reuters_NOOW.csv      wiki_ref220_natural_order.csv
MKB10.csv	 RTL_Wiki.csv


In [22]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/20NG.csv',
)

dataset.get_possible_modalities()

{'@bigram', '@lemmatized'}

In [23]:
MAIN_MODALITY = '@lemmatized'

In [24]:
dataset._data.head()

,Unnamed: 0,raw_text,filenames,target,id,tokenized,lemmatized,bigram,vw_text
id,,,,,,,,,
rec_autos_102994,0,I was wondering if anyone out there could enli...,/home/egorov/scikit_learn_data/20news_home/20n...,7,rec_autos_102994,"[('was', 'VBD'), ('wondering', 'VBG'), ('if', ...","['wonder', 'anyone', 'could', 'enlighten', 'ca...","['wonder_anyone', 'anyone_could', 'sport_car',...",rec_autos_102994 |@lemmatized wonder:1 anyone:...
comp_sys_mac_hardware_51861,1,A fair number of brave souls who upgraded thei...,/home/egorov/scikit_learn_data/20news_home/20n...,4,comp_sys_mac_hardware_51861,"[('fair', 'JJ'), ('number', 'NN'), ('of', 'IN'...","['fair', 'number', 'brave', 'soul', 'upgrade',...","['clock_oscillator', 'please_send', 'top_speed...",comp_sys_mac_hardware_51861 |@lemmatized fair:...
comp_sys_mac_hardware_51879,2,"well folks, my mac plus finally gave up the gh...",/home/egorov/scikit_learn_data/20news_home/20n...,4,comp_sys_mac_hardware_51879,"[('well', 'RB'), ('folks', 'NNS'), ('my', 'PRP...","['well', 'folk', 'mac', 'plus', 'finally', 'gi...","['mac_plus', 'life_way', 'way_back', 'market_n...",comp_sys_mac_hardware_51879 |@lemmatized well:...
comp_graphics_38242,3,\nDo you have Weitek's address/phone number? ...,/home/egorov/scikit_learn_data/20news_home/20n...,1,comp_graphics_38242,"[('do', 'VBP'), ('you', 'PRP'), ('have', 'VB')...","['weitek', 'address', 'phone', 'number', 'like...","['address_phone', 'phone_number', 'number_like...",comp_graphics_38242 |@lemmatized weitek:1 addr...
sci_space_60880,4,"From article <C5owCB.n3p@world.std.com>, by to...",/home/egorov/scikit_learn_data/20news_home/20n...,14,sci_space_60880,"[('from', 'IN'), ('article', 'NN'), ('by', 'IN...","['article', 'tom', 'baker', 'understanding', '...","['system_software', 'thing_check', 'introduce_...",sci_space_60880 |@lemmatized article:1 tom:1 b...


In [25]:
dataset._data.shape

(11301, 9)

In [26]:
dataset._data.dropna(axis=0, inplace=True)

In [27]:
dataset._data.shape

(11083, 9)

In [28]:
dataset._data['raw_text']

id
rec_autos_102994                  I was wondering if anyone out there could enli...
comp_sys_mac_hardware_51861       A fair number of brave souls who upgraded thei...
comp_sys_mac_hardware_51879       well folks, my mac plus finally gave up the gh...
comp_graphics_38242               \nDo you have Weitek's address/phone number?  ...
sci_space_60880                   From article <C5owCB.n3p@world.std.com>, by to...
                                                        ...                        
sci_med_58069                     DN> From: nyeda@cnsvax.uwec.edu (David Nye)\nD...
comp_sys_mac_hardware_51712       I have a (very old) Mac 512k and a Mac Plus, b...
comp_sys_ibm_pc_hardware_60695    I just installed a DX2-66 CPU in a clone mothe...
comp_graphics_38319               \nWouldn't this require a hyper-sphere.  In 3-...
rec_motorcycles_104440            Stolen from Pasadena between 4:30 and 6:30 pm ...
Name: raw_text, Length: 11083, dtype: object

In [29]:
docs = list(dataset._data['raw_text'].values)

In [30]:
docs[:3]

['I was wondering if anyone out there could enlighten me on this car I saw\nthe other day. It was a 2-door sports car, looked to be from the late 60s/\nearly 70s. It was called a Bricklin. The doors were really small. In addition,\nthe front bumper was separate from the rest of the body. This is \nall I know. If anyone can tellme a model name, engine specs, years\nof production, where this car is made, history, or whatever info you\nhave on this funky looking car, please e-mail.',
 "A fair number of brave souls who upgraded their SI clock oscillator have\nshared their experiences for this poll. Please send a brief message detailing\nyour experiences with the procedure. Top speed attained, CPU rated speed,\nadd on cards and adapters, heat sinks, hour of usage per day, floppy disk\nfunctionality with 800 and 1.4 m floppies are especially requested.\n\nI will be summarizing in the next two days, so please add to the network\nknowledge base if you have done the clock upgrade and haven't an

In [31]:
NUM_TOP_WORDS = 20

In [32]:
import torch
import transformers
import os

import json
import numpy as np

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [33]:
from torch import bfloat16
import transformers

# set quantization configuration to load large model with less GPU memory
# this requires the `bitsandbytes` library

bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,  # 4-bit quantization
    bnb_4bit_quant_type='nf4',  # Normalized float 4
    bnb_4bit_use_double_quant=True,  # Second quantization after the first
    bnb_4bit_compute_dtype=bfloat16  # Computation type
)

In [35]:
"""<s>You are a friendly chatbot who always responds in the style of a pirate<|end_of_turn|>GPT4 Correct User: Hello, my name is<|end_of_turn|>GPT4 Correct Assistant: Charlie<|end_of_turn|>GPT4 Correct User: How are you, Charlie?<|end_of_turn|>GPT4 Correct Assistant: 
"""

'<s>You are a friendly chatbot who always responds in the style of a pirate<|end_of_turn|>GPT4 Correct User: Hello, my name is<|end_of_turn|>GPT4 Correct Assistant: Charlie<|end_of_turn|>GPT4 Correct User: How are you, Charlie?<|end_of_turn|>GPT4 Correct Assistant: \n'

In [47]:
def get_phi(topic_model):
    mtw = topic_model.c_tf_idf_.todense()
    # ptw = np.array(mtw[1:, :])
    ptw = np.array(mtw[0:, :])
    pwt = ptw.T
    vocabulary = topic_model.vectorizer_model.get_feature_names_out()

    assert pwt.shape[0] == len(vocabulary)

    # topic_names = [f'topic_{i}' for i in range(pwt.shape[1])]
    topic_names = ['background_1'] + [f'topic_{i}' for i in range(NUM_TOPICS)]
    phi = pd.DataFrame(
        index=vocabulary,
        columns=topic_names,
        data=pwt,
    )

    return phi


def get_top_words(topic_model):
    mtw = topic_model.c_tf_idf_.todense()
    ptw = np.array(mtw[0:, :])
    pwt = ptw.T

    topic_names = ['background_1'] + [f'topic_{i}' for i in range(NUM_TOPICS)]
    topic_top_words = {
        n: topic_model.get_topic(t)
        for t, n in zip([-1] + list(range(NUM_TOPICS)), topic_names)
    }

    return topic_top_words


def get_dataset(topic_model, dataset, docs):
    cleaned_docs = topic_model._preprocess_text(docs)
    vectorizer = topic_model.vectorizer_model
    tokenizer = vectorizer.build_tokenizer()
    doc_tokens = [tokenizer(doc) for doc in cleaned_docs]
    doc_texts = [
        d + f' |{MAIN_MODALITY} ' + ' '.join(t)
        for d, t in zip(dataset._data.index, doc_tokens)
    ]
    data = [[d, t] for d, t in zip(dataset._data.index, doc_texts)]

    new_dataset = pd.DataFrame(
        columns=['id', 'vw_text'],
        data=data,
    )

    return new_dataset

In [39]:
NUM_TOPICS = 50
NUM_TOP_WORDS = 20
NUM_TRAINS = 20
STOP_WORDS = 'english'
LANGUAGE = 'english'

In [41]:
! ls ../results50

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [43]:
SAVE_FOLDER = os.path.join('/data_mil/shared/CompressaAI/BERTopic', 'results50', '20newsgroups')

In [44]:
! mkdir -p $SAVE_FOLDER

In [45]:
SAVE_FOLDER

'/data_mil/shared/CompressaAI/BERTopic/results50/20newsgroups'

In [50]:
for seed in range(NUM_TRAINS):
    print(seed)

    seed_save_folder = os.path.join(SAVE_FOLDER, str(seed))

    os.makedirs(seed_save_folder)

    keybert = KeyBERTInspired(top_n_words=NUM_TOP_WORDS)
    mmr = MaximalMarginalRelevance(diversity=0.3, top_n_words=NUM_TOP_WORDS)
    
    representation_model = {
        "KeyBERT": keybert,
        "MMR": mmr,
    }

    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=seed)
    vectorizer_model = CountVectorizer(stop_words=STOP_WORDS)

    topic_model = BERTopic(
        language=LANGUAGE,
        top_n_words=NUM_TOP_WORDS,

        calculate_probabilities=True,
        verbose=True,
    
        umap_model=umap_model,                    # Step 2 - Reduce dimensionality
        # hdbscan_model=hdbscan_model,            # Step 3 - Cluster reduced embeddings
        vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
        representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
    )
        
    topics, probs = topic_model.fit_transform(docs)
    orig_num_topics = len(set(topic_model.topics_))
    doc_embeddings = topic_model.umap_model.embedding_

    hdbscan_model = HDBSCAN_flat(doc_embeddings, n_clusters=NUM_TOPICS)
    
    topic_model = BERTopic(
        language=LANGUAGE,
        top_n_words=NUM_TOP_WORDS,
        calculate_probabilities=True,
        verbose=True,
    
        umap_model=umap_model,                    # Step 2 - Reduce dimensionality
        hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
        vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
        representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
    )

    topics, probs = topic_model.fit_transform(docs)
    
    new_num_topics = len(set(topic_model.topics_))
    
    assert new_num_topics < orig_num_topics
    assert new_num_topics == NUM_TOPICS + 1
    assert topic_model.c_tf_idf_.shape[0] == new_num_topics
    
    phi = get_phi(topic_model)
    top_words = get_top_words(topic_model)
    new_dataset = get_dataset(topic_model, dataset, docs)
    
    phi.to_csv(f'{seed_save_folder}/phi.csv')
    
    with open(f'{seed_save_folder}/top_words.json', 'w') as f:
        f.write(
            json.dumps(
                top_words, indent=4, ensure_ascii=False
            )
        )
    
    new_dataset.to_csv(f'{seed_save_folder}/dataset.csv')

    del topic_model, phi, new_dataset

2024-03-30 00:23:57,797 - BERTopic - Embedding - Transforming documents to embeddings.


0


Batches:   0%|          | 0/347 [00:00<?, ?it/s]

2024-03-30 00:24:11,966 - BERTopic - Embedding - Completed ✓
2024-03-30 00:24:11,967 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 00:24:20,774 - BERTopic - Dimensionality - Completed ✓
2024-03-30 00:24:20,775 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 00:24:29,241 - BERTopic - Cluster - Completed ✓
2024-03-30 00:24:29,246 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 00:24:41,675 - BERTopic - Representation - Completed ✓
2024-03-30 00:24:44,989 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/347 [00:00<?, ?it/s]

...

19


Batches:   0%|          | 0/347 [00:00<?, ?it/s]

2024-03-30 01:06:26,892 - BERTopic - Embedding - Completed ✓
2024-03-30 01:06:26,893 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 01:06:35,611 - BERTopic - Dimensionality - Completed ✓
2024-03-30 01:06:35,613 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 01:06:43,876 - BERTopic - Cluster - Completed ✓
2024-03-30 01:06:43,881 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 01:06:56,210 - BERTopic - Representation - Completed ✓
2024-03-30 01:06:59,388 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/347 [00:00<?, ?it/s]

2024-03-30 01:07:13,793 - BERTopic - Embedding - Completed ✓
2024-03-30 01:07:13,795 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 01:07:22,678 - BERTopic - Dimensionality - Completed ✓
2024-03-30 01:07:22,679 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 01:08:10,854 - BERTopic - Cluster - Completed ✓
2024-03-30 01:08:10,859 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 01:08:17,251 - BERTopic - Representation - Completed ✓
